In [1]:
import sqlite3
import pandas as pd

In [3]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

In [4]:
cursor.executescript('''
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    username TEXT NOT NULL,
    created_at DATE NOT NULL
);

CREATE TABLE user_activity (
    id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    activity_type_id INTEGER NOT NULL,
    activity_date DATE NOT NULL,
    FOREIGN KEY (user_id) REFERENCES users(id)
);

CREATE TABLE activity_types (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE user_roles (
    id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    role TEXT NOT NULL,
    assigned_at DATE NOT NULL,
    FOREIGN KEY (user_id) REFERENCES users(id)
);
''')

In [5]:
cursor.executescript('''
INSERT INTO users (id, username, created_at) VALUES
(1, 'user1', '2024-01-01'),
(2, 'user2', '2024-02-15'),
(3, 'user3', '2024-03-10'),
(4, 'user4', '2024-04-01'),
(5, 'user5', '2024-05-01'),
(6, 'user6', '2024-06-01');

INSERT INTO activity_types (id, name) VALUES
(1, 'login'),
(2, 'logout'),
(3, 'purchase');

INSERT INTO user_activity (id, user_id, activity_type_id, activity_date) VALUES
(1, 1, 1, '2024-10-01'),
(2, 1, 2, '2024-10-05'),
(3, 1, 1, '2024-10-10'),
(4, 2, 1, '2024-10-15'),
(5, 2, 3, '2024-09-20'),
(6, 3, 1, '2024-08-25'),
(7, 4, 1, '2024-10-22'),
(8, 4, 2, '2024-10-25'),
(9, 6, 1, '2024-10-05'),
(10, 6, 3, '2024-10-10'),
(11, 6, 1, '2024-09-30');

INSERT INTO user_roles (id, user_id, role, assigned_at) VALUES
(1, 1, 'admin', '2024-10-01'),
(2, 1, 'moderator', '2024-10-05'),
(3, 2, 'user', '2024-10-10'),
(4, 4, 'guest', '2024-10-20'),
(5, 6, 'editor', '2024-10-15');
''')

conn.commit()

In [24]:
query = '''
WITH last_month_activity AS (
SELECT user_id, COUNT(*) as num_activities
FROM user_activity
WHERE activity_date >= '2024-10-01'
GROUP BY user_id
),
users_last_month AS (
SELECT user_id, username, num_activities
FROM last_month_activity LEFT JOIN users
ON users.id = last_month_activity.user_id
)
SELECT username, GROUP_CONCAT(role, ','), num_activities
FROM users_last_month LEFT JOIN user_roles
ON users_last_month.user_id = user_roles.user_id
GROUP BY user_roles.user_id, username
ORDER BY num_activities DESC
'''
pd.read_sql_query(query, conn)

,username,"GROUP_CONCAT(role, ',')",num_activities
0,user1,"admin,moderator",3
1,user4,guest,2
2,user6,editor,2
3,user2,user,1


In [25]:
conn.close()